# Model Evaluation, Explainability, and Bias Auditing
**Objective:** Evaluate our trained XGBoost model, interpret its decisions using SHAP, and perform a fairness audit across demographic groups.

In [ ]:
from pathlib import Path

import pandas as pd
import numpy as np
import joblib
import shap
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, PrecisionRecallDisplay

# Resolve paths from either the project root or the notebooks/ directory
candidate_roots = [Path.cwd(), Path.cwd().parent]
project_root = next(
    (root for root in candidate_roots if (root / 'models' / 'xgb_fraud_model.pkl').exists()),
    Path.cwd().parent,
 )

# Load processed test data and the trained model
X_test = pd.read_csv(project_root / 'data' / 'processed' / 'X_test.csv')
y_test = pd.read_csv(project_root / 'data' / 'processed' / 'y_test.csv').squeeze()
model = joblib.load(project_root / 'models' / 'xgb_fraud_model.pkl')

/opt/anaconda3/lib/python3.12/site-packages/pandas/core/computation/expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/opt/anaconda3/lib/python3.12/site-packages/pandas/core/arrays/masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


In [ ]:
# 1. Evaluate Model Performance (PR-AUC)
y_prob = model.predict_proba(X_test)[:, 1]

display = PrecisionRecallDisplay.from_predictions(
    y_test, y_prob, name="XGBoost", color="darkorange"
)
_ = display.ax_.set_title("Precision-Recall Curve")

### Step 5: Explainability (SHAP)
We use SHAP values to understand which features are driving the model to flag a transaction as fraud. This is critical for regulatory compliance in finance.

In [ ]:
# 2. SHAP Feature Importance
# Using a sample of the test set for faster computation
X_test_sample = X_test.sample(1000, random_state=42)
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test_sample)

# Plot summary
shap.summary_plot(shap_values, X_test_sample, plot_type="bar")

### Step 5: Bias Detection & Fairness Audit
To fulfill our ethical AI requirements, we will simulate a fairness audit. We will assign a synthetic 'Age_Group' to our test set to see if our model's False Positive Rate (FPR) unfairly penalizes a specific demographic (e.g., older users).

In [ ]:
# 3. Fairness Audit (Simulated)
# Assign a random age group to our test data for demonstration purposes
np.random.seed(42)
age_groups = np.random.choice(['Under 30', '30-55', 'Over 55'], size=len(y_test))

# Create an evaluation dataframe
eval_df = pd.DataFrame({
    'True_Class': y_test,
    'Predicted_Class': model.predict(X_test),
    'Age_Group': age_groups
})

# Calculate False Positive Rate (FPR) for each group
# FPR = False Positives / (False Positives + True Negatives)
def calculate_fpr(group_df):
    fp = len(group_df[(group_df['Predicted_Class'] == 1) & (group_df['True_Class'] == 0)])
    tn = len(group_df[(group_df['Predicted_Class'] == 0) & (group_df['True_Class'] == 0)])
    return fp / (fp + tn) if (fp + tn) > 0 else 0

print("--- Fairness Audit: False Positive Rate by Age Group ---")
for group in ['Under 30', '30-55', 'Over 55']:
    group_data = eval_df[eval_df['Age_Group'] == group]
    fpr = calculate_fpr(group_data)
    print(f"Age Group '{group}': FPR = {fpr:.4f} ({fpr*100:.2f}%)")

print("\nConclusion: If the FPR varies significantly across groups, we must investigate mitigation strategies (e.g., threshold adjustments) to ensure Equalized Odds.")